In [1]:
import json
import os
import glob
import shutil
import sys

from pathlib import Path
from typing import List, Optional, Dict, Tuple
from logger import _setup_child_logger, _setup_root_logger

from post_process_merge_jsons import(
    CLIMJSONMerger,
    MOVSJSONMerger,
    ENSOJSONMerger,
)

# Set up the root logger and module level logger. The module level logger is
# a child of the root logger.
_setup_root_logger()
logger = _setup_child_logger(__name__)


In [2]:
class PCMDJSONMerger:
    """
    Orchestrate JSON merging for mean climate, variability modes, and ENSO metrics.
    """

    def __init__(
        self,
        pmprdir: str | Path,
        mips: Optional[List[str]] = None,
        exps: Optional[List[str]] = None,
        member_dirs: Optional[List[str]] = None,
        model_rename: Optional[List[str]] = None,
        *,
        case_id_clim: Optional[str] = None,
        out_path_clim: Optional[str] = None,
        case_id_movs: Optional[str] = None,
        out_path_movs: Optional[str] = None,
        case_id_enso: Optional[str] = None,
        out_path_enso: Optional[str] = None,
        movs_years: Tuple[int, int] = (1900, 2014),
        movs_obses: Optional[str] = None,
        movs_modes: Optional[str] = None,
        enso_collections: Optional[str] = None,
        enso_obses: Optional[str] = None,
        enable_clim: bool = True,
        enable_movs: bool = True,
        enable_enso: bool = True,
        strict: bool = False,
        verbose: bool = True,
        dry_run: bool = False,
        enso_output_type: str = "metrics_results",
    ) -> None:
        self.pmprdir = Path(pmprdir)
        self.mips = mips or ["cmip6"]
        self.exps = exps or ["historical"]
        self.member_dirs = member_dirs or []
        self.model_rename = model_rename or None 
        
        self.case_id_clim = case_id_clim
        self.case_id_movs = case_id_movs
        self.case_id_enso = case_id_enso

        self.out_path_clim = out_path_clim
        self.out_path_movs = out_path_movs
        self.out_path_enso = out_path_enso

        self.movs_years = movs_years
        self.movs_obses = movs_obses
        self.movs_modes = movs_modes
        
        self.enso_collections = enso_collections
        self.enso_obses = enso_obses
        
        self.enable_clim = enable_clim
        self.enable_movs = enable_movs
        self.enable_enso = enable_enso

        self.strict = strict
        self.verbose = verbose
        self.dry_run = dry_run
        self.enso_output_type = enso_output_type

    # --------------------------
    # Internal utilities
    # --------------------------
    def _log(self, *msg) -> None:
        if self.verbose:
            print(*msg)

    # --------------------------
    # Public entry points
    # --------------------------
    def run_all(self) -> None:
        if self.enable_clim:
            self.run_clim()
        if self.enable_movs:
            self.run_movs()
        if self.enable_enso:
            self.run_enso()

    def run_clim(self) -> None:
        if not self.case_id_clim:
            self._log("[CLIM] No case_id provided — skipping.")
            return
        if not self.out_path_clim:
            self._log("[CLIM][WARN] out_path_clim not set; using pmprdir as output root.")
            self.out_path_clim = str(self.pmprdir)

        self._log(f"[CLIM] Merging: mips={self.mips} exps={self.exps} case_id={self.case_id_clim}")

        # Each member dir is the metrics_data root; CLIMJSONMerger expects the leaf 'mean_climate'
        member_meanclim_dirs = [f"{d}/mean_climate" for d in self.member_dirs]

        CLIMJSONMerger(
            mips=self.mips,
            exps=self.exps,
            case_id=self.case_id_clim,
            model_rename=self.model_rename,
            pmprdir=str(self.pmprdir),
            out_path=self.out_path_clim,          # IMPORTANT: this should be the PARENT data root
            member_dirs=member_meanclim_dirs,
            strict=self.strict,
            verbose=self.verbose,
            dry_run=self.dry_run,
        ).merge_all()

    def run_movs(self) -> None:
        if not self.case_id_movs:
            self._log("[MOVS] No case_id provided — skipping.")
            return
        if not self.out_path_movs:
            self._log("[MOVS][WARN] out_path_movs not set; using pmprdir as output root.")
            self.out_path_movs = str(self.pmprdir)

        syear, eyear = self.movs_years
        self._log(
            f"[MOVS] Merging: mips={self.mips} exps={self.exps} case_id={self.case_id_movs} "
            f"years={syear}-{eyear} obs={self.movs_obses} mode={self.movs_modes}"
        )
        movs_obses = list(self.movs_obses.split(","))
        movs_modes = list(self.movs_modes.split(","))
        if len(movs_obses) != len(movs_modes):
            self._log("[MOVS][ERROR] movs_obses not match movs_modes; please check...")

        member_movs_dirs = [f"{d}/variability_modes" for d in self.member_dirs]

        MOVSJSONMerger(
            mips=self.mips,
            exps=self.exps,
            case_id=self.case_id_movs,
            model_rename=self.model_rename,
            pmprdir=str(self.pmprdir),
            out_path=self.out_path_movs,       
            member_dirs=member_movs_dirs,
            movs_obses=movs_obses,
            movs_modes=movs_modes,
            syear=syear,
            eyear=eyear,
            strict=self.strict,
            verbose=self.verbose,
            dry_run=self.dry_run,
        ).merge_all()
        
    def run_enso(self) -> None:
        if not self.case_id_enso:
            self._log("[ENSO] No case_id provided — skipping.")
            return
        if not self.out_path_enso:
            self._log("[ENSO][WARN] out_path_movs not set; using pmprdir as output root.")
            self.out_path_enso = str(self.pmprdir)
        
        collections = list(self.enso_collections.split(","))
        enso_obses = list(self.enso_obses.split(","))
        if len(enso_obses) != len(collections):
            self._log("[ENSO][ERROR] enso_obses not match collections; please check...")

        member_enso_dirs = [f"{d}/enso_metric" for d in self.member_dirs]
        
        ENSOJSONMerger(
            mips=self.mips,
            exps=self.exps,
            case_id=self.case_id_enso,
            model_rename=self.model_rename,
            pmprdir=str(self.pmprdir),
            out_path=self.out_path_enso,       
            member_dirs=member_enso_dirs,
            collections=collections,
            observations=enso_obses,
            strict=self.strict,
            verbose=self.verbose,
            dry_run=self.dry_run,
        ).merge_all()


In [5]:
if __name__ == "__main__":
    """Main function to drive the entire process."""
    # Configuration
    run_type = 'model_vs_obs'
    diag_path = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/e3sm-pcmdi-le"
        
    enable_clim  = True
    enable_movs  = True
    enable_enso  = True
    
    mips = ["e3sm"]
    exps = ["historical"]
    model_pattern = "v3.LR.historical"
    
    for group in ["climo","future","historical"]:
        ##########################################
        if group == "climo":
            pmpr_base = os.path.join(diag_path,'climo')
            print(pmpr_base)
            ##########################################
            # renmae (old,new,with_ensemble)
            ##########################################
            model_rename = ("v3-LR", "v3-LR-Historical", True) 
            # customize reference set
            ref_clim_dir = "/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/mean_climate"
            ref_movs_dir = "/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/variability_modes"
            ref_enso_dir = "/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/enso_metric"
            case_id_clim = "v20260216"
            case_id_movs = "v20260216"
            case_id_enso = "v20260216"
            
            movs_years   = (1850,2014)
            movs_modes   = "NAM,NAO,PNA,NPO,SAM,PSA1,PSA2,PDO,NPGO,AMO"
            movs_obses   = "NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,HadISST,HadISST,HadISST"
        
            enso_collections = "ENSO_perf,ENSO_tel,ENSO_proc"
            enso_obses = "ERA-Interim,ERA-Interim,ERA-Interim"
            enso_output_type = "metrics_results"  # or "metrics_data" for refs
            
        elif group == "historical":
            pmpr_base = os.path.join(diag_path,'hist')
            ##########################################
            # renmae (old,new,with_ensemble)
            ##########################################
            model_rename = ("v3-LR", "v3-LR-Historical", True) 
            # customize reference set
            ref_clim_dir = "/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/mean_climate"
            ref_movs_dir = "/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/variability_modes"
            ref_enso_dir = "/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/enso_metric"
            case_id_clim = "v20260216"
            case_id_movs = "v20260216"
            case_id_enso = "v20260216"
            
            movs_years   = (1980,2014)
            movs_modes   = "NAM,NAO,PNA,NPO,SAM,PSA1,PSA2,PDO,NPGO,AMO"
            movs_obses   = "NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,HadISST,HadISST,HadISST"
        
            enso_collections = "ENSO_perf,ENSO_tel,ENSO_proc"
            enso_obses = "ERA-Interim,ERA-Interim,ERA-Interim"
            enso_output_type = "metrics_results"  # or "metrics_data" for refs
        else:
            pmpr_base = os.path.join(diag_path,group)
            ##########################################
            # renmae (old,new,with_ensemble)
            ##########################################
            model_rename = ("v3-LR", "v3-LR-Future", True) 
            # customize reference set
            ref_clim_dir = "/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/mean_climate"
            ref_movs_dir = "/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/variability_modes"
            ref_enso_dir = "/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/enso_metric"
            case_id_clim = "v20260216"
            case_id_movs = "v20260216"
            case_id_enso = "v20260216"
            
            movs_years   = (2015,2049)
            movs_modes   = "NAM,NAO,PNA,NPO,SAM,PSA1,PSA2,PDO,NPGO,AMO"
            movs_obses   = "NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,HadISST,HadISST,HadISST"
            
            enso_collections = "ENSO_perf,ENSO_tel,ENSO_proc"
            enso_obses = "FAKE-FUTURE1,FAKE-FUTURE1,FAKE-FUTURE1"
            enso_output_type = "metrics_results"  # or "metrics_data" for refs
            
        pattern = os.path.join(pmpr_base, f"{model_pattern}_*")
        model_names = sorted([os.path.basename(d) for d in glob.glob(pattern) if os.path.isdir(d)])
        member_dirs = [f"{pmpr_base}/{mname}/pcmdi_diags/model_vs_obs/metrics_data" for mname in model_names] 
        
        strict=True
        verbose=True
        dry_run=False

        # clean up the directory to ensure clean process
        for mip in mips:
            for exp in exps:
                for odir, cid in zip(
                    [ref_clim_dir, ref_movs_dir, ref_enso_dir],
                    [case_id_clim, case_id_movs, case_id_enso],
                ):
                    outpath = os.path.join(odir, mip, exp, cid)
                    tgtpath = os.path.join(odir, mip, group, cid)
        
                    outpath_n = os.path.realpath(outpath)
                    tgtpath_n = os.path.realpath(tgtpath)
        
                    if os.path.exists(outpath_n):
                        if os.path.isdir(outpath_n):
                            shutil.rmtree(outpath_n)
                        else:
                            os.remove(outpath_n)
        
                    if os.path.exists(tgtpath_n):
                        if os.path.isdir(tgtpath_n):
                            shutil.rmtree(tgtpath_n)
                        else:
                            os.remove(tgtpath_n)
                                
                        
        merger = PCMDJSONMerger(
            pmprdir=pmpr_base,
            mips=mips,                 # or ["cmip6"] depending on your tree
            exps=exps,
            member_dirs=member_dirs,
            model_rename=model_rename,
            case_id_clim=case_id_clim,      # from your main block
            out_path_clim=ref_clim_dir,
            case_id_movs=case_id_movs,
            out_path_movs=ref_movs_dir,
            case_id_enso=case_id_enso,      # set None to auto-detect latest
            out_path_enso=ref_enso_dir,
            movs_years=movs_years,
            movs_modes=movs_modes,
            movs_obses=movs_obses,
            enso_collections=enso_collections,
            enso_obses=enso_obses,
            enable_clim=enable_clim,
            enable_movs=enable_movs,
            enable_enso=enable_enso,
            strict=strict,
            verbose=verbose,
            dry_run=dry_run,
            enso_output_type=enso_output_type,  # or "metrics_data" for refs
        )
        
        merger.run_all()
        
        # move the data to correct directory (replace if exists, no-op if same)
        for mip in mips:
            for exp in exps:
                for odir, cid in zip(
                    [ref_clim_dir, ref_movs_dir, ref_enso_dir],
                    [case_id_clim, case_id_movs, case_id_enso],
                ):
                    outpath = os.path.join(odir, mip, exp, cid)
                    tgtpath = os.path.join(odir, mip, group, cid)
        
                    # normalize paths to avoid false mismatches
                    outpath_n = os.path.realpath(outpath)
                    tgtpath_n = os.path.realpath(tgtpath)
        
                    if outpath_n == tgtpath_n:
                        print(outpath_n)
                        print(tgtpath_n)
                        continue
        
                    if not os.path.exists(outpath_n):
                        continue
        
                    # remove existing target (dir or file)
                    if os.path.exists(tgtpath_n):
                        if os.path.isdir(tgtpath_n):
                            shutil.rmtree(tgtpath_n)
                        else:
                            os.remove(tgtpath_n)
        
                    # ensure parent directory exists
                    os.makedirs(os.path.dirname(tgtpath_n), exist_ok=True)
        
                    shutil.move(outpath_n, tgtpath_n)
        
                    # --- rename files inside tgtpath (recursive) ---
                    for root, _, files in os.walk(tgtpath_n):
                        for fname in files:
                            if exp not in fname:
                                continue
                            old = os.path.join(root, fname)
                            new = os.path.join(root, fname.replace(exp, group))
        
                            # if a file with the new name already exists, replace it
                            if os.path.exists(new):
                                os.remove(new)
        
                            os.rename(old, new)
        
                    print(outpath_n)
                    print(tgtpath_n)

/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/e3sm-pcmdi-le/climo
[CLIM] Merging: mips=['e3sm'] exps=['historical'] case_id=v20260216
[CLIM] e3sm/historical variables: []
[MOVS] Merging: mips=['e3sm'] exps=['historical'] case_id=v20260216 years=1850-2014 obs=NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,HadISST,HadISST,HadISST mode=NAM,NAO,PNA,NPO,SAM,PSA1,PSA2,PDO,NPGO,AMO
[MOVS] discovered 0 (mode,obs,mip,exp,eof) combos
[INFO] mip=e3sm exp=historical mc=ENSO_perf obs=ERA-Interim case_id=v20260216
[ENSO] Merging 25 files: e3sm/historical/ENSO_perf/ERA-Interim
  [1/25] /lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/e3sm-pcmdi-le/climo/v3.LR.historical_0051/pcmdi_diags/model_vs_obs/metrics_data/enso_metric/ENSO_perf/ENSO_perf.e3sm.historical.v3-LR_0051.vs.ERA-Interim.v20260216.json
  [2/25] /lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/e3sm-pcmdi-le/climo/v3.LR.historical_0091/pcmdi_diags/model_vs_obs/metrics_data/enso_metric/ENSO_perf/E

In [16]:
if __name__ == "__main__":
    """Main function to drive the entire process."""
    # Configuration
    run_type = 'model_vs_obs'
    diag_path = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/e3sm-pcmdi-le"
        
    enable_clim  = True
    enable_movs  = True
    enable_enso  = True
    
    mips = ["e3sm"]
    exps = ["historical"]
    model_pattern = "v3.LR.historical"
    
    for group in ["climo","future","historical"]:
        ##########################################
        if group == "climo":
            pmpr_base = os.path.join(diag_path,'climo')
            print(pmpr_base)
            ##########################################
            # renmae (old,new,with_ensemble)
            ##########################################
            model_rename = ("v3-LR", "v3-LR-Historical", True) 
            # customize reference set
            ref_clim_dir = "/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/mean_climate"
            ref_movs_dir = "/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/variability_modes"
            ref_enso_dir = "/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/enso_metric"
            case_id_clim = "v20260212"
            case_id_movs = "v20260212"
            case_id_enso = "v20260212"
            
            movs_years   = (1850,2014)
            movs_modes   = "NAM,NAO,PNA,NPO,SAM,PSA1,PSA2,PDO,NPGO,AMO"
            movs_obses   = "NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,HadISST,HadISST,HadISST"
        
            enso_collections = "ENSO_perf,ENSO_tel,ENSO_proc"
            enso_obses = "Tropflux,Tropflux,Tropflux"
            enso_output_type = "metrics_results"  # or "metrics_data" for refs
            
        elif group == "historical":
            pmpr_base = os.path.join(diag_path,'hist')
            ##########################################
            # renmae (old,new,with_ensemble)
            ##########################################
            model_rename = ("v3-LR", "v3-LR-Historical", True) 
            # customize reference set
            ref_clim_dir = "/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/mean_climate"
            ref_movs_dir = "/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/variability_modes"
            ref_enso_dir = "/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/enso_metric"
            case_id_clim = "v20260212"
            case_id_movs = "v20260212"
            case_id_enso = "v20260212"
            
            movs_years   = (1980,2014)
            movs_modes   = "NAM,NAO,PNA,NPO,SAM,PSA1,PSA2,PDO,NPGO,AMO"
            movs_obses   = "NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,HadISST,HadISST,HadISST"
        
            enso_collections = "ENSO_perf,ENSO_tel,ENSO_proc"
            enso_obses = "Tropflux,Tropflux,Tropflux"
            enso_output_type = "metrics_results"  # or "metrics_data" for refs
        else:
            pmpr_base = os.path.join(diag_path,group)
            ##########################################
            # renmae (old,new,with_ensemble)
            ##########################################
            model_rename = ("v3-LR", "v3-LR-Future", True) 
            # customize reference set
            ref_clim_dir = "/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/mean_climate"
            ref_movs_dir = "/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/variability_modes"
            ref_enso_dir = "/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/enso_metric"
            case_id_clim = "v20260212"
            case_id_movs = "v20260212"
            case_id_enso = "v20260212"
            
            movs_years   = (2015,2049)
            movs_modes   = "NAM,NAO,PNA,NPO,SAM,PSA1,PSA2,PDO,NPGO,AMO"
            movs_obses   = "NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,HadISST,HadISST,HadISST"
            
            enso_collections = "ENSO_perf,ENSO_tel,ENSO_proc"
            enso_obses = "FAKE-FUTURE2,FAKE-FUTURE2,FAKE-FUTURE2"
            enso_output_type = "metrics_results"  # or "metrics_data" for refs
            
        pattern = os.path.join(pmpr_base, f"{model_pattern}_*")
        model_names = sorted([os.path.basename(d) for d in glob.glob(pattern) if os.path.isdir(d)])
        member_dirs = [f"{pmpr_base}/{mname}/pcmdi_diags/model_vs_obs/metrics_data" for mname in model_names] 
        
        strict=True
        verbose=True
        dry_run=False

        # clean up the directory to ensure clean process
        for mip in mips:
            for exp in exps:
                for odir, cid in zip(
                    [ref_clim_dir, ref_movs_dir, ref_enso_dir],
                    [case_id_clim, case_id_movs, case_id_enso],
                ):
                    outpath = os.path.join(odir, mip, exp, cid)
                    tgtpath = os.path.join(odir, mip, group, cid)
        
                    outpath_n = os.path.realpath(outpath)
                    tgtpath_n = os.path.realpath(tgtpath)
        
                    if os.path.exists(outpath_n):
                        if os.path.isdir(outpath_n):
                            shutil.rmtree(outpath_n)
                        else:
                            os.remove(outpath_n)
        
                    if os.path.exists(tgtpath_n):
                        if os.path.isdir(tgtpath_n):
                            shutil.rmtree(tgtpath_n)
                        else:
                            os.remove(tgtpath_n)
                                
                        
        merger = PCMDJSONMerger(
            pmprdir=pmpr_base,
            mips=mips,                 # or ["cmip6"] depending on your tree
            exps=exps,
            member_dirs=member_dirs,
            model_rename=model_rename,
            case_id_clim=case_id_clim,      # from your main block
            out_path_clim=ref_clim_dir,
            case_id_movs=case_id_movs,
            out_path_movs=ref_movs_dir,
            case_id_enso=case_id_enso,      # set None to auto-detect latest
            out_path_enso=ref_enso_dir,
            movs_years=movs_years,
            movs_modes=movs_modes,
            movs_obses=movs_obses,
            enso_collections=enso_collections,
            enso_obses=enso_obses,
            enable_clim=enable_clim,
            enable_movs=enable_movs,
            enable_enso=enable_enso,
            strict=strict,
            verbose=verbose,
            dry_run=dry_run,
            enso_output_type=enso_output_type,  # or "metrics_data" for refs
        )
        
        merger.run_all()
        
        # move the data to correct directory (replace if exists, no-op if same)
        for mip in mips:
            for exp in exps:
                for odir, cid in zip(
                    [ref_clim_dir, ref_movs_dir, ref_enso_dir],
                    [case_id_clim, case_id_movs, case_id_enso],
                ):
                    outpath = os.path.join(odir, mip, exp, cid)
                    tgtpath = os.path.join(odir, mip, group, cid)
        
                    # normalize paths to avoid false mismatches
                    outpath_n = os.path.realpath(outpath)
                    tgtpath_n = os.path.realpath(tgtpath)
        
                    if outpath_n == tgtpath_n:
                        print(outpath_n)
                        print(tgtpath_n)
                        continue
        
                    if not os.path.exists(outpath_n):
                        continue
        
                    # remove existing target (dir or file)
                    if os.path.exists(tgtpath_n):
                        if os.path.isdir(tgtpath_n):
                            shutil.rmtree(tgtpath_n)
                        else:
                            os.remove(tgtpath_n)
        
                    # ensure parent directory exists
                    os.makedirs(os.path.dirname(tgtpath_n), exist_ok=True)
        
                    shutil.move(outpath_n, tgtpath_n)
        
                    # --- rename files inside tgtpath (recursive) ---
                    for root, _, files in os.walk(tgtpath_n):
                        for fname in files:
                            if exp not in fname:
                                continue
                            old = os.path.join(root, fname)
                            new = os.path.join(root, fname.replace(exp, group))
        
                            # if a file with the new name already exists, replace it
                            if os.path.exists(new):
                                os.remove(new)
        
                            os.rename(old, new)
        
                    print(outpath_n)
                    print(tgtpath_n)
                

/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/e3sm-pcmdi-le/climo
[CLIM] Merging: mips=['e3sm'] exps=['historical'] case_id=v20260212
[CLIM] e3sm/historical variables: ['pr', 'prw', 'psl', 'rlds', 'rldscs', 'rltcre', 'rlus', 'rlut', 'rlutcs', 'rsds', 'rsdscs', 'rsdt', 'rstcre', 'rsus', 'rsuscs', 'rsut', 'rsutcs', 'rtmt', 'sfcWind', 'ta-200', 'ta-850', 'tas', 'tauu', 'tauv', 'ts', 'ua-200', 'ua-850', 'va-200', 'va-850', 'zg-500']
[CLIM] Search in: [members x25] var=pr
[CLIM] 25 files matched for var=pr
  [1/25] /lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/e3sm-pcmdi-le/climo/v3.LR.historical_0051/pcmdi_diags/model_vs_obs/metrics_data/mean_climate/pr.2.5x2.5.e3sm.historical.v3-LR_0051.v20260212.json
  [2/25] /lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/e3sm-pcmdi-le/climo/v3.LR.historical_0091/pcmdi_diags/model_vs_obs/metrics_data/mean_climate/pr.2.5x2.5.e3sm.historical.v3-LR_0091.v20260212.json
  [3/25] /lcrc/group/e3sm/public_html/diagnostic_output/ac

In [14]:
if __name__ == "__main__":
    """Main function to drive the entire process."""
    # Configuration
    run_type = 'model_vs_obs'
    diag_path = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/e3sm-pcmdi-le"
        
    enable_clim  = True
    enable_movs  = True
    enable_enso  = True
    
    mips = ["e3sm"]
    exps = ["historical"]
    model_pattern = "v3.LR.historical"
    
    for group in ["climo","future","historical"]:
        ##########################################
        if group == "climo":
            pmpr_base = os.path.join(diag_path,'climo')
            print(pmpr_base)
            ##########################################
            # renmae (old,new,with_ensemble)
            ##########################################
            model_rename = ("v3-LR", "v3-LR-Historical", True) 
            # customize reference set
            ref_clim_dir = "/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/mean_climate"
            ref_movs_dir = "/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/variability_modes"
            ref_enso_dir = "/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/enso_metric"
            case_id_clim = "v20251015"
            case_id_movs = "v20251015"
            case_id_enso = "v20251015"
            
            movs_years   = (1850,2014)
            movs_modes   = "NAM,NAO,PNA,NPO,SAM,PSA1,PSA2,PDO,NPGO,AMO"
            movs_obses   = "NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,HadISST,HadISST,HadISST"
        
            enso_collections = "ENSO_perf,ENSO_tel,ENSO_proc"
            enso_obses = "ERA-Interim,ERA-Interim,ERA-Interim"
            enso_output_type = "metrics_results"  # or "metrics_data" for refs
        elif group == "historical":
            pmpr_base = os.path.join(diag_path,'hist')
            ##########################################
            # renmae (old,new,with_ensemble)
            ##########################################
            model_rename = ("v3-LR", "v3-LR-Historical", True) 
            # customize reference set
            ref_clim_dir = "/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/mean_climate"
            ref_movs_dir = "/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/variability_modes"
            ref_enso_dir = "/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/enso_metric"
            case_id_clim = "v20251015"
            case_id_movs = "v20251015"
            case_id_enso = "v20251015"
            
            movs_years   = (1980,2014)
            movs_modes   = "NAM,NAO,PNA,NPO,SAM,PSA1,PSA2,PDO,NPGO,AMO"
            movs_obses   = "NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,HadISST,HadISST,HadISST"
        
            enso_collections = "ENSO_perf,ENSO_tel,ENSO_proc"
            enso_obses = "ERA-Interim,ERA-Interim,ERA-Interim"
            enso_output_type = "metrics_results"  # or "metrics_data" for refs
        else:
            pmpr_base = os.path.join(diag_path,group)
            ##########################################
            # renmae (old,new,with_ensemble)
            ##########################################
            model_rename = ("v3-LR", "v3-LR-Future", True) 
            # customize reference set
            ref_clim_dir = "/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/mean_climate"
            ref_movs_dir = "/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/variability_modes"
            ref_enso_dir = "/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/enso_metric"
            case_id_clim = "v20251015"
            case_id_movs = "v20251015"
            case_id_enso = "v20251015"
            
            movs_years   = (2015,2049)
            movs_modes   = "NAM,NAO,PNA,NPO,SAM,PSA1,PSA2,PDO,NPGO,AMO"
            movs_obses   = "NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,HadISST,HadISST,HadISST"
            
            enso_collections = "ENSO_perf,ENSO_tel,ENSO_proc"
            enso_obses = "ERA-Interim2,ERA-Interim2,ERA-Interim2"
            enso_output_type = "metrics_results"  # or "metrics_data" for refs
            
        pattern = os.path.join(pmpr_base, f"{model_pattern}_*")
        model_names = sorted([os.path.basename(d) for d in glob.glob(pattern) if os.path.isdir(d)])
        member_dirs = [f"{pmpr_base}/{mname}/pcmdi_diags/model_vs_obs/metrics_data" for mname in model_names] 
        
        strict=True
        verbose=True
        dry_run=False
        # clean up the directory to ensure clean process
        for mip in mips:
            for exp in exps:
                for odir, cid in zip(
                    [ref_clim_dir, ref_movs_dir, ref_enso_dir],
                    [case_id_clim, case_id_movs, case_id_enso],
                ):
                    outpath = os.path.join(odir, mip, exp, cid)
                    tgtpath = os.path.join(odir, mip, group, cid)
        
                    outpath_n = os.path.realpath(outpath)
                    tgtpath_n = os.path.realpath(tgtpath)
        
                    if os.path.exists(outpath_n):
                        if os.path.isdir(outpath_n):
                            shutil.rmtree(outpath_n)
                        else:
                            os.remove(outpath_n)
        
                    if os.path.exists(tgtpath_n):
                        if os.path.isdir(tgtpath_n):
                            shutil.rmtree(tgtpath_n)
                        else:
                            os.remove(tgtpath_n)
                            
        merger = PCMDJSONMerger(
            pmprdir=pmpr_base,
            mips=mips,                 # or ["cmip6"] depending on your tree
            exps=exps,
            member_dirs=member_dirs,
            model_rename=model_rename,
            case_id_clim=case_id_clim,      # from your main block
            out_path_clim=ref_clim_dir,
            case_id_movs=case_id_movs,
            out_path_movs=ref_movs_dir,
            case_id_enso=case_id_enso,      # set None to auto-detect latest
            out_path_enso=ref_enso_dir,
            movs_years=movs_years,
            movs_modes=movs_modes,
            movs_obses=movs_obses,
            enso_collections=enso_collections,
            enso_obses=enso_obses,
            enable_clim=enable_clim,
            enable_movs=enable_movs,
            enable_enso=enable_enso,
            strict=strict,
            verbose=verbose,
            dry_run=dry_run,
            enso_output_type=enso_output_type,  # or "metrics_data" for refs
        )
        
        merger.run_all()
        
        # move the data to correct directory (replace if exists, no-op if same)
        for mip in mips:
            for exp in exps:
                for odir, cid in zip(
                    [ref_clim_dir, ref_movs_dir, ref_enso_dir],
                    [case_id_clim, case_id_movs, case_id_enso],
                ):
                    outpath = os.path.join(odir, mip, exp, cid)
                    tgtpath = os.path.join(odir, mip, group, cid)
        
                    # normalize paths to avoid false mismatches
                    outpath_n = os.path.realpath(outpath)
                    tgtpath_n = os.path.realpath(tgtpath)
        
                    if outpath_n == tgtpath_n:
                        print(outpath_n)
                        print(tgtpath_n)
                        continue
        
                    if not os.path.exists(outpath_n):
                        continue
        
                    # remove existing target (dir or file)
                    if os.path.exists(tgtpath_n):
                        if os.path.isdir(tgtpath_n):
                            shutil.rmtree(tgtpath_n)
                        else:
                            os.remove(tgtpath_n)
        
                    # ensure parent directory exists
                    os.makedirs(os.path.dirname(tgtpath_n), exist_ok=True)
        
                    shutil.move(outpath_n, tgtpath_n)
        
                    # --- rename files inside tgtpath (recursive) ---
                    for root, _, files in os.walk(tgtpath_n):
                        for fname in files:
                            if exp not in fname:
                                continue
                            old = os.path.join(root, fname)
                            new = os.path.join(root, fname.replace(exp, group))
        
                            # if a file with the new name already exists, replace it
                            if os.path.exists(new):
                                os.remove(new)
        
                            os.rename(old, new)
        
                    print(outpath_n)
                    print(tgtpath_n)
                

/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/e3sm-pcmdi-le/climo
[CLIM] Merging: mips=['e3sm'] exps=['historical'] case_id=v20251015
[CLIM] e3sm/historical variables: ['pr', 'prw', 'psl', 'rlds', 'rldscs', 'rltcre', 'rlus', 'rlut', 'rlutcs', 'rsds', 'rsdscs', 'rsdt', 'rstcre', 'rsus', 'rsuscs', 'rsut', 'rsutcs', 'rtmt', 'sfcWind', 'ta-200', 'ta-850', 'tas', 'tauu', 'tauv', 'ts', 'ua-200', 'ua-850', 'va-200', 'va-850', 'zg-500']
[CLIM] Search in: [members x25] var=pr
[CLIM] 24 files matched for var=pr
  [1/24] /lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/e3sm-pcmdi-le/climo/v3.LR.historical_0051/pcmdi_diags/model_vs_obs/metrics_data/mean_climate/pr.2.5x2.5.e3sm.historical.v3-LR_0051.v20251015.json
  [2/24] /lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/e3sm-pcmdi-le/climo/v3.LR.historical_0091/pcmdi_diags/model_vs_obs/metrics_data/mean_climate/pr.2.5x2.5.e3sm.historical.v3-LR_0091.v20251015.json
  [3/24] /lcrc/group/e3sm/public_html/diagnostic_output/ac